# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [26]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania: 3.55s


In [28]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania (wątki): 0.77s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [20]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [21]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.35s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania. 

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [1]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
LICZBA_ZAPYTAN = 20

def pobierz_fakt(_=None):
    try:
        response = requests.get(CAT_API_URL, timeout=5)
        return response.json().get('fact')
    except Exception:
        return None

def pobieranie_sekwencyjne():
    fakty = []
    for _ in range(LICZBA_ZAPYTAN):
        fakt = pobierz_fakt()
        if fakt:
            fakty.append(fakt)
    return fakty

def pobieranie_wielowatkowe():
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        wyniki = list(executor.map(pobierz_fakt, range(LICZBA_ZAPYTAN)))
    return [f for f in wyniki if f is not None]

if __name__ == "__main__":
    start_seq = time.time()
    fakty_seq = pobieranie_sekwencyjne()
    koniec_seq = time.time()
    czas_seq = koniec_seq - start_seq

    start_thread = time.time()
    fakty_thread = pobieranie_wielowatkowe()
    koniec_thread = time.time()
    czas_thread = koniec_thread - start_thread

    print(f"Pobrano sekwencyjnie: {len(fakty_seq)} faktów w czasie: {czas_seq:.2f} s")
    print(f"Pobrano wielowatkowo: {len(fakty_thread)} faktów w czasie: {czas_thread:.2f} s")
    print(f"Wielowatkowosc byla {czas_seq / czas_thread:.1f}x szybsza.")

Pobrano sekwencyjnie: 20 faktów w czasie: 9.61 s
Pobrano wielowatkowo: 20 faktów w czasie: 2.00 s
Wielowatkowosc byla 4.8x szybsza.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [2]:
import queue
import threading
import time

PULA_LICZB = 20
kolejka = queue.Queue()

def producent():
    for i in range(1, PULA_LICZB + 1):
        kolejka.put(i)
        time.sleep(0.01)
    kolejka.put(None)
    kolejka.put(None)

def konsument_parzyste():
    while True:
        liczba = kolejka.get()
        if liczba is None:
            kolejka.task_done()
            break
        if liczba % 2 == 0:
            print(f"[Konsument 1 - Parzyste] Pobrałem: {liczba}")
            kolejka.task_done()
        else:
            kolejka.put(liczba)
            kolejka.task_done()
            time.sleep(0.01)

def konsument_nieparzyste():
    while True:
        liczba = kolejka.get()
        if liczba is None:
            kolejka.task_done()
            break
        if liczba % 2 != 0:
            print(f"[Konsument 2 - Nieparzyste] Pobrałem: {liczba}")
            kolejka.task_done()
        else:
            kolejka.put(liczba)
            kolejka.task_done()
            time.sleep(0.01)

if __name__ == "__main__":
    watek_producenta = threading.Thread(target=producent)
    watek_konsumenta_1 = threading.Thread(target=konsument_parzyste)
    watek_konsumenta_2 = threading.Thread(target=konsument_nieparzyste)

    watek_producenta.start()
    watek_konsumenta_1.start()
    watek_konsumenta_2.start()

    watek_producenta.join()
    watek_konsumenta_1.join()
    watek_konsumenta_2.join()

[Konsument 2 - Nieparzyste] Pobrałem: 1
[Konsument 1 - Parzyste] Pobrałem: 2
[Konsument 2 - Nieparzyste] Pobrałem: 3
[Konsument 1 - Parzyste] Pobrałem: 4
[Konsument 2 - Nieparzyste] Pobrałem: 5
[Konsument 1 - Parzyste] Pobrałem: 6
[Konsument 2 - Nieparzyste] Pobrałem: 7
[Konsument 1 - Parzyste] Pobrałem: 8
[Konsument 2 - Nieparzyste] Pobrałem: 9
[Konsument 1 - Parzyste] Pobrałem: 10
[Konsument 2 - Nieparzyste] Pobrałem: 11
[Konsument 1 - Parzyste] Pobrałem: 12
[Konsument 2 - Nieparzyste] Pobrałem: 13
[Konsument 1 - Parzyste] Pobrałem: 14
[Konsument 2 - Nieparzyste] Pobrałem: 15
[Konsument 1 - Parzyste] Pobrałem: 16
[Konsument 2 - Nieparzyste] Pobrałem: 17
[Konsument 1 - Parzyste] Pobrałem: 18
[Konsument 2 - Nieparzyste] Pobrałem: 19
[Konsument 1 - Parzyste] Pobrałem: 20


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [3]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    zakres_liczb = list(range(1, 10001))

    start_seq = time.time()
    wyniki_seq = [calculate_power_sum(n) for n in zakres_liczb]
    koniec_seq = time.time()
    czas_seq = koniec_seq - start_seq
    print(f"Obliczenia sekwencyjne: {czas_seq:.4f} s")

    start_multi = time.time()
    liczba_procesorow = multiprocessing.cpu_count()
    with multiprocessing.Pool(processes=liczba_procesorow) as pool:
        wyniki_multi = pool.map(calculate_power_sum, zakres_liczb)
    koniec_multi = time.time()
    czas_multi = koniec_multi - start_multi
    print(f"Obliczenia wieloprocesowe ({liczba_procesorow} rdzeni): {czas_multi:.4f} s")

    print(f"Przyspieszenie: {czas_seq / czas_multi:.1f}x")

Obliczenia sekwencyjne: 0.4295 s
Obliczenia wieloprocesowe (12 rdzeni): 0.3423 s
Przyspieszenie: 1.3x
